# <center> **Proyecto visión por computador:** Análisis exploratorio de datos </center>

**Por:**
- Alejandro Olivares
- Santiago Zapata
- Alejandra Galeano
- María José Bernal

## 1. Librerías

In [ ]:
import os
import sys
import hashlib
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

import cv2
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from PIL import Image
from matplotlib.image import imread
from matplotlib.colors import LinearSegmentedColormap

%config InlineBackend.figure_formats = ["jpeg"]
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
from config import ARTIFACTS_DIR, DATA_DIR, SPLIT_CSV

EDA_DIR = ARTIFACTS_DIR / "eda"
EDA_DIR.mkdir(parents = True, exist_ok = True)


def guardar(fig, nombre):
    fig.savefig(EDA_DIR / f"{nombre}.png", dpi = 120, bbox_inches = "tight")


print(f"Datos: {DATA_DIR}")
print(f"Figuras y tablas del EDA: {EDA_DIR}")

## 2. Lectura y transformación de datos

Primero se cargarán los datos almacenados en la carpeta `TRAINING_PATH`, correspondientes al conjunto de entrenamiento del [dataset de Kaggle](https://www.kaggle.com/competitions/understanding_cloud_organization/data), ya que estos son los que contienen la información de la segmentación de cada una de las clases.

El dataset proviene de la competencia de Kaggle. En Kaggle se lee directamente de los datos de la competencia y, en otro equipo, de la carpeta `data/` del repositorio (ver `src/config.py`):
- `train.csv`: metadatos con una fila por cada par imagen-clase. `Image_Label` tiene el formato `<imagen>_<clase>` y `EncodedPixels` contiene la máscara en formato RLE (pares inicio-longitud, con índice desde 1 y recorrido por columnas); queda vacío si la clase no está presente.
- `train_images/`: imágenes satelitales en formato JPG.

Las cuatro clases son patrones de organización de nubes:
- **Sugar:** nubes dispersas de pequeña escala, con aspecto de polvo.
- **Gravel:** líneas o arcos de pequeñas celdas nubosas con bordes definidos.
- **Flower:** grandes masas de nubes con forma de flor y zonas despejadas entre ellas.
- **Fish:** estructuras alargadas con forma de espina de pescado.

In [ ]:
TRAINING_PATH = DATA_DIR / "train.csv"

training_df = pd.read_csv(TRAINING_PATH)
training_df.head()

Ahora, se dividirá la columna `Image_Label` en las columnas de `Image` (nombre del archivo) y `Label` (etiqueta de clase).

In [ ]:
training_df[["Image", "Label"]] = training_df["Image_Label"].str.split("_", expand = True)
training_df = training_df[["Image", "Label", "EncodedPixels"]]
training_df.fillna("", inplace = True)
training_df.head()

Luego, dado que una misma imagen puede contener más de un tipo de nube, se pivotea el dataframe para que cada fila corresponda a una imagen y cada columna a una clase, con los píxeles donde se encuentra esa clase en la imagen en formato RLE (Run-Length Encoding).

In [ ]:
pivot_training_df = training_df.pivot(index = "Image", columns = "Label", values = "EncodedPixels")
pivot_training_df.head()

Finalmente, se guarda el nuevo dataframe en un archivo CSV para facilitar su consulta posterior.

In [ ]:
OUTPUT_TRAINING_PATH = ROOT / "output" / "training_pivot.csv"
os.makedirs(os.path.dirname(OUTPUT_TRAINING_PATH), exist_ok = True)
pivot_training_df.to_csv(OUTPUT_TRAINING_PATH, index = True, sep = ",")

## 3. Análisis exploratorio de datos (EDA)
Una vez organizada la estructura del conjunto de datos, se realizará un análisis exploratorio con el propósito de identificar la distribución de las clases, evaluar posibles desbalances, analizar la coexistencia de etiquetas dentro de una misma imagen y caracterizar la cobertura espacial de las máscaras de segmentación. Este proceso permitió obtener una visión más profunda de los datos y establecer criterios para las etapas posteriores de preprocesamiento, entrenamiento y evaluación de los modelos. Las secciones siguen el orden del checklist de exploración de datos.

In [ ]:
df = pd.read_csv(OUTPUT_TRAINING_PATH)
df.head()

### 3.1. Estructura del dataset

In [ ]:
LABELS = df.columns[1:]

presence_df = (pivot_training_df != "").astype(int)
presence_df.head()

In [ ]:
IMAGES_PATH = str(DATA_DIR / "train_images")
image_files = sorted(os.listdir(IMAGES_PATH))
csv_images = set(df["Image"])

print(f"Imágenes en la carpeta: {len(image_files)}")
print(f"Imágenes únicas en el dataframe: {len(csv_images)}")
print(f"Dimensiones del dataframe: {len(df)} imágenes x {len(LABELS)} clases")
print(f"Imágenes del dataframe sin archivo: {len(csv_images - set(image_files))}")
print(f"Archivos sin registro en el dataframe: {len(set(image_files) - csv_images)}")
print(f"Clases: {list(LABELS)}")

El conjunto de entrenamiento tiene 5546 imágenes. En el dataframe pivoteado cada imagen es una fila y cada clase una columna. Todas las imágenes del dataframe tienen su archivo y no hay archivos sin registro, por lo que la estructura es consistente.

### 3.2. Estadísticas básicas
Se recorren todas las imágenes una sola vez para obtener su tamaño, un hash del archivo (para detectar duplicados) y estadísticas de color y nitidez calculadas a un cuarto de la resolución. Estas estadísticas se reutilizan en las secciones siguientes.


In [ ]:
def get_image_info(img_name):
    path = os.path.join(IMAGES_PATH, img_name)
    with open(path, "rb") as f:
        info = {"Image": img_name, "md5": hashlib.md5(f.read()).hexdigest()}
    try:
        with Image.open(path) as img:
            info["width"], info["height"] = img.size
            img.draft("RGB", (img.width // 4, img.height // 4))
            pixels = np.asarray(img.convert("RGB"), dtype = np.float32) / 255
    except Exception:
        return {**info, "ok": False}

    gray = pixels.mean(axis = 2)
    laplacian = gray[:-2, 1:-1] + gray[2:, 1:-1] + gray[1:-1, :-2] + gray[1:-1, 2:] - 4 * gray[1:-1, 1:-1]
    for i, channel in enumerate("RGB"):
        info[f"{channel}_mean"] = pixels[..., i].mean()
        info[f"{channel}_std"] = pixels[..., i].std()
    return {**info, "ok": True, "brightness": gray.mean(), "sharpness": laplacian.var()}


with ThreadPoolExecutor() as executor:
    image_df = pd.DataFrame(executor.map(get_image_info, image_files)).set_index("Image")
image_df.head()

In [ ]:
print(image_df.groupby(["width", "height"]).size().rename("imágenes").to_string(), end = "\n\n")
print(f"Relación de aspecto: {(image_df['width'] / image_df['height']).round(2).unique()}")
print(f"Imágenes con contenido duplicado: {image_df['md5'].duplicated(keep = False).sum()}")
print(f"\nImágenes por clase:\n{presence_df.sum().to_string()}")

In [ ]:
IMG_HEIGHT, IMG_WIDTH = 1400, 2100
total_pixels = IMG_HEIGHT * IMG_WIDTH

Todas las imágenes miden 2100x1400 píxeles (relación de aspecto 3:2) y no hay contenido duplicado, por lo que no se requiere eliminar muestras redundantes.

### 3.3. Visualización de imágenes
Para verificar la calidad general de los datos, se realizó una inspección visual de imágenes seleccionadas aleatoriamente junto con sus respectivas máscaras de segmentación.

In [ ]:
def rle_to_mask(rle_string, height, width):
    if rle_string == "":
        return np.zeros((height, width))
    nums = [int(x) for x in rle_string.split()]
    pairs = np.array(nums).reshape(-1, 2)
    mask = np.zeros(height * width, dtype=np.uint8)
    for index, length in pairs:
        index -= 1
        mask[index:index + length] = 255
    return mask.reshape(width, height).T / 255.0

In [ ]:
def draw_label(mask, label, ax, label_color):
    if np.sum(mask) > 0:
        y, x = np.argwhere(mask == 1)[0]
        ax.text(x + 10, y + 40, label,
                bbox = dict(fill = True, color = label_color))

In [ ]:
def plot_image(img_name, images_path, ax = None):
    label_colors = {
        "Fish":   "#2171b5",
        "Flower": "#238b45",
        "Gravel": "#cb181d",
        "Sugar":  "#7b2d8b"
    }
    cmaps = {
        label: LinearSegmentedColormap.from_list(label, [color, color])
        for label, color in label_colors.items()
    }

    img = imread(os.path.join(images_path, img_name))
    h, w = img.shape[:2]

    standalone = ax is None
    if standalone:
        fig, ax = plt.subplots(figsize = (8, 6))

    ax.imshow(img)
    for label, cmap in cmaps.items():
        mask = rle_to_mask(pivot_training_df.loc[img_name, label], h, w)
        mask = np.clip(mask, 0, 1)
        masked = np.ma.masked_where(mask == 0, mask)
        ax.imshow(masked, alpha = 0.4, cmap = cmap)
        draw_label(mask, label, ax, label_colors[label])

    ax.set_title(img_name, fontweight = "bold")
    ax.axis("off")

    if standalone:
        plt.tight_layout()
        plt.show()

#### 3.3.1. Muestras aleatorias


In [ ]:
images_path = IMAGES_PATH

sample = pivot_training_df[
    (pivot_training_df != "").any(axis = 1)
].sample(4, random_state = 44)

fig, axes = plt.subplots(2, 2, figsize = (16, 12))
for ax, img_name in zip(axes.flat, sample.index):
    plot_image(img_name, images_path, ax = ax)
plt.tight_layout()
plt.show()

#### 3.3.2. Ejemplos de cada clase
Se muestra una imagen en la que aparece únicamente la clase indicada.


In [ ]:
single_label = presence_df[presence_df.sum(axis = 1) == 1]

fig, axes = plt.subplots(1, 4, figsize = (20, 4))
for ax, label in zip(axes, LABELS):
    img_name = single_label[single_label[label] == 1].sample(1, random_state = 44).index[0]
    plot_image(img_name, IMAGES_PATH, ax = ax)
    ax.set_title(f"{label}: {img_name}", fontweight = "bold")
plt.tight_layout()
plt.show()

Adicionalmente, se identificaron características visuales distintivas para cada categoría. Las nubes tipo Fish presentan estructuras alargadas y filamentosas; Flower muestra patrones más circulares y difusos; Gravel se caracteriza por texturas fragmentadas e irregulares; mientras que Sugar exhibe distribuciones granulares más finas y dispersas. Estas diferencias morfológicas sugieren que las clases poseen características visuales suficientemente diferenciables para ser aprendidas por modelos de segmentación basados en aprendizaje profundo.

#### 3.3.3. Casos límite
Se inspeccionan las imágenes más oscura, más clara, menos nítida y más nítida del conjunto, para detectar posibles outliers.


In [ ]:
outliers = {
    "Más oscura": image_df["brightness"].idxmin(),
    "Más clara": image_df["brightness"].idxmax(),
    "Menos nítida": image_df["sharpness"].idxmin(),
    "Más nítida": image_df["sharpness"].idxmax(),
}

fig, axes = plt.subplots(2, 2, figsize = (12, 8))
for ax, (reason, img_name) in zip(axes.flat, outliers.items()):
    plot_image(img_name, IMAGES_PATH, ax = ax)
    ax.set_title(f"{reason}: {img_name}", fontweight = "bold")
plt.tight_layout()
plt.show()

Los casos límite corresponden a imágenes con grandes franjas negras sin datos (por ejemplo, la más oscura y menos nítida) y a imágenes muy claras o sobreexpuestas. Son casos aislados, ya que el brillo mínimo es 0.06 frente a una mediana de 0.28, y conviene conservarlos porque representan la variabilidad real del conjunto.

Además, en las imágenes mostradas las máscaras son regiones rectangulares que aproximan la zona de cada patrón, pero no siguen el contorno exacto de las nubes. Esto limita la precisión que se puede esperar en los bordes.

### 3.4. Distribución de clases y exploración de etiquetas

#### 3.4.1. Frecuencia de cada clase
Se examina en cuántas imágenes aparece cada una de las cuatro categorías. El porcentaje se calcula sobre el total de imágenes, por lo que no suma 100 %: una misma imagen puede contener varias clases.

In [ ]:
color_map = dict(zip(["Fish", "Flower", "Gravel", "Sugar"], ["#03396c", "#005b96", "#6497b1", "#b3cde0"]))
text_color = {"Fish": "white", "Flower": "white", "Gravel": "black", "Sugar": "black"}
class_counts = presence_df.sum().sort_values()
n_images = len(presence_df)
bar_colors = [color_map[label] for label in class_counts.index]

fig, ax = plt.subplots(figsize = (8, 5))
bars = ax.barh(class_counts.index, class_counts.values, color = bar_colors)

for bar, label in zip(bars, class_counts.index):
    count = int(bar.get_width())
    pct = count / n_images * 100
    ax.text(bar.get_width() - class_counts.max() * 0.01, bar.get_y() + bar.get_height() / 2,
            f"{count} ({pct:.1f} % de las imágenes)", ha = "right", va = "center", fontsize = 10,
            color = text_color[label])

ax.set_title("Frecuencia de clases en las imágenes satelitales", fontweight = "bold")
ax.set_xlabel("Cantidad de imágenes")
ax.set_ylabel("Tipo de nube")
plt.tight_layout()
guardar(fig, "frecuencia_clases")
plt.show()

Sugar aparece en el 67.6 % de las imágenes, Gravel en el 53.0 %, Fish en el 50.1 % y Flower en el 42.6 %. La distribución es desigual, pues Flower es la clase menos representada. Este desbalance puede afectar el aprendizaje del modelo al favorecer las categorías con mayor representación.

Por eso conviene usar estrategias de mitigación durante el entrenamiento, como funciones de pérdida especializadas para segmentación (la pérdida focal da más peso a los píxeles difíciles) y aumento de datos que mejore la generalización en las categorías menos representadas.

#### 3.4.2. Distribución del número de etiquetas
Debido a que una misma imagen puede contener múltiples tipos de nubes simultáneamente, se analizó la cantidad de etiquetas presentes en cada imagen.

In [ ]:
label_counts = presence_df.sum(axis = 1).value_counts().sort_index()

fig, ax = plt.subplots(figsize = (7, 5))
bars = ax.bar(label_counts.index, label_counts.values, color = "#5470b0")

for bar in bars:
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 10,
            str(int(bar.get_height())), ha = "center", va = "bottom", fontsize = 10)
    
ax.set_title("Distribución de etiquetas por imagen satelital", fontweight = "bold")
ax.set_xlabel("Número de etiquetas")
ax.set_ylabel("Número de imágenes")
ax.set_xticks(label_counts.index)
plt.tight_layout()
guardar(fig, "etiquetas_por_imagen")
plt.show()

De la gráfica anterior se observa que la mayoría de las imágenes contienen entre una y tres clases activas, siendo dos el caso más común (2372 imágenes), mientras que las imágenes con las cuatro clases son poco frecuentes (266) y no existen imágenes sin ninguna etiqueta. Este comportamiento confirma la naturaleza multi-etiqueta del problema y evidencia que el modelo deberá aprender a identificar y segmentar múltiples patrones nubosos dentro de una misma escena satelital.

#### 3.4.3. Co-ocurrencia entre etiquetas
Con el fin de estudiar las relaciones existentes entre los distintos patrones de nubes, se construyó una matriz de co-ocurrencia que cuantifica cuántas imágenes contienen simultáneamente cada par de etiquetas.

In [ ]:
co_occurence_mtx = presence_df.T @ presence_df

fig, ax = plt.subplots(figsize = (6, 5))
sns.heatmap(co_occurence_mtx, annot = True, fmt = "d", cmap = "Blues", ax = ax, linewidths = 0.5, square = True)
ax.set_title("Co-ocurrencia de etiquetas", fontweight = "bold")
plt.tight_layout()
guardar(fig, "coocurrencia")
plt.show()

La pareja más habitual es Sugar–Gravel (2067 imágenes), seguida de Sugar–Fish (1782) y Sugar–Flower (1405), y la menos frecuente es Flower–Gravel (926). Sin embargo, estos conteos dependen en buena parte de la frecuencia de cada clase: Sugar está en dos de cada tres imágenes, así que es esperable que aparezca con todas. Para separar ese efecto de una relación real entre clases, se compara cada conteo con el que se esperaría si las clases aparecieran de forma independiente.

Si dos clases fueran independientes, el número esperado de imágenes con ambas sería

$$E_{ij} = \frac{n_i \, n_j}{N}$$

donde $n_i$ y $n_j$ son las imágenes que contienen cada clase y $N$ es el total de imágenes. La razón entre lo observado y lo esperado indica si dos clases aparecen juntas más (razón mayor que 1) o menos (razón menor que 1) de lo que dicta el azar.

In [ ]:
n_per_class = presence_df.sum()
expected_mtx = np.outer(n_per_class, n_per_class) / len(presence_df)
ratio = co_occurence_mtx.to_numpy(dtype = float) / expected_mtx
np.fill_diagonal(ratio, np.nan)
ratio_mtx = pd.DataFrame(ratio, index = co_occurence_mtx.index, columns = co_occurence_mtx.columns)

fig, ax = plt.subplots(figsize = (6, 5))
sns.heatmap(ratio_mtx, annot = True, fmt = ".2f", cmap = "RdBu", center = 1, vmin = 0.6, vmax = 1.4,
            ax = ax, linewidths = 0.5, square = True)
ax.set_title("Co-ocurrencia observada / esperada", fontweight = "bold")
plt.tight_layout()
guardar(fig, "coocurrencia_vs_independencia")
plt.show()

Todas las razones están entre 0.74 y 1.04, es decir, cerca de lo que daría el azar. La pareja más frecuente, Sugar–Gravel, aparece apenas un 4 % más de lo esperado (2067 frente a unas 1988), así que su conteo alto se explica casi por completo porque ambas clases son frecuentes y no por condiciones atmosféricas compartidas. La relación más marcada es la de Flower con Gravel, que aparecen juntas un 26 % menos de lo esperado (926 frente a unas 1253). En general, Flower coincide con las demás clases menos de lo que dicta el azar (razones entre 0.74 y 0.88), lo que sugiere que tiende a dominar la escena cuando está presente.

#### 3.4.4. Correlación de etiquetas
Adicionalmente, se calculó la correlación de Pearson entre las variables binarias que indican la presencia o ausencia de cada clase en las imágenes.

In [ ]:
fig, ax = plt.subplots(figsize = (6, 5))
sns.heatmap(presence_df.corr(), annot = True, fmt = ".2f", cmap = "Blues",
            ax = ax, vmin = -1, vmax = 1, linewidths = 0.5, square = True)
ax.set_title("Correlación de etiquetas", fontweight = "bold")
plt.tight_layout()
guardar(fig, "correlacion")
plt.show()

La matriz de correlación confirma lo anterior. Las relaciones lineales entre las clases son bajas, por lo que la presencia de una categoría no predice con fuerza la aparición de otra. La correlación más fuerte es la negativa entre Flower y Gravel (−0.24), seguida de Flower–Sugar (−0.15) y Fish–Flower (−0.10), mientras que las demás están entre −0.08 y 0.06. Es la misma lectura de la razón observada/esperada: la única relación que se aleja del azar de forma apreciable es que Flower y Gravel tienden a no aparecer juntas.

#### 3.4.5. Representación de las clases
El dataset no incluye subclases ni atributos adicionales (como edad o género) que analizar. Sugar es la clase más frecuente (3751 imágenes, 68 %) y Flower la menos frecuente (2365 imágenes, 43 %), una relación de aproximadamente 1.6 a 1. Por lo tanto, no hay subrepresentación ni sobrerrepresentación extrema, sino un desbalance moderado.


### 3.5. Calidad de los datos
Se verifica que las imágenes se puedan leer, que no haya imágenes de baja calidad (se usa la varianza del laplaciano como medida de nitidez) y que las máscaras RLE sean consistentes: índices dentro de la imagen, longitudes positivas y segmentos ordenados y sin solapamiento.


In [ ]:
def is_valid_rle(rle):
    if rle == "":
        return True
    nums = np.array(rle.split(), dtype = int)
    starts, lengths = nums[0::2], nums[1::2]
    ends = starts + lengths - 1
    return bool((starts >= 1).all() and (lengths > 0).all() and (ends <= total_pixels).all() and (starts[1:] > ends[:-1]).all())


invalid_rle = {label: int((~pivot_training_df[label].map(is_valid_rle)).sum()) for label in LABELS}
wrong_size = (image_df["width"] != IMG_WIDTH) | (image_df["height"] != IMG_HEIGHT)
low_sharpness = image_df["sharpness"] < 0.1 * image_df["sharpness"].median()

print(f"Imágenes corruptas o ilegibles: {(~image_df['ok']).sum()}")
print(f"Imágenes con tamaño distinto a {IMG_WIDTH}x{IMG_HEIGHT}: {wrong_size.sum()}")
print(f"Imágenes con nitidez menor al 10% de la mediana: {low_sharpness.sum()}")
print(f"Máscaras RLE inválidas por clase: {invalid_rle}")
image_df[["brightness", "sharpness"]].describe().round(4)

No se encontraron imágenes corruptas, con tamaño distinto ni con nitidez anómala, y todas las máscaras RLE son válidas (dentro de la imagen, con segmentos ordenados y sin solapamiento interno). Las franjas negras sin datos de algunas imágenes no son un error de lectura, pero sí una fuente de ruido propia del dataset.

### 3.6. Características de las imágenes

#### 3.6.1. Intensidad de los píxeles
Se analiza el brillo promedio y la media de cada canal de color por imagen, para identificar sesgos de iluminación.


In [ ]:
print(f"Brillo promedio: {image_df['brightness'].mean():.4f}")
print(f"Desviación estándar del brillo: {image_df['brightness'].std():.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize = (12, 4))

axes[0].hist(image_df["brightness"], bins = 40, color = "#5470b0", edgecolor = "white")
axes[0].set_title("Brillo promedio por imagen", fontweight = "bold")
axes[0].set_xlabel("Brillo (0-1)")
axes[0].set_ylabel("Número de Imágenes")

for channel, color in zip("RGB", ["#cb181d", "#238b45", "#2171b5"]):
    axes[1].hist(image_df[f"{channel}_mean"], bins = 40, alpha = 0.6, color = color, label = channel)
axes[1].set_title("Media por canal de color", fontweight = "bold")
axes[1].set_xlabel("Media del canal (0-1)")
axes[1].legend()

plt.tight_layout()
plt.show()

El brillo promedio es 0.29 (desviación de 0.05), con una distribución sesgada hacia valores altos y casos aislados muy oscuros por las franjas negras. En cuanto al color, el azul tiene la media más alta, seguido del verde y el rojo, lo que es coherente con escenas de océano y cielo.

#### 3.6.2. Área de la máscara por imagen
Se evaluó el porcentaje del área de cada imagen cubierto por al menos una máscara de segmentación.

In [ ]:
def rle_pixel_count(rle_string):
    if rle_string == "":
        return 0
    nums = list(map(int, rle_string.split()))
    return sum(nums[1::2])

In [ ]:
def unique_pixel_count(row):
    mask = np.zeros(total_pixels, dtype=bool)
    for label in LABELS:
        rle = row[label]
        if rle == "":
            continue
        nums = np.array(rle.split(), dtype=int)
        starts = nums[0::2] - 1
        lengths = nums[1::2]
        for s, l in zip(starts, lengths):
            mask[s:s + l] = True
    return mask.sum()

In [ ]:
coverage_pixels = pd.DataFrame(
    {label: pivot_training_df[label].map(rle_pixel_count) for label in LABELS}
)

total_coverage_pct = pivot_training_df[LABELS].apply(unique_pixel_count, axis = 1) / total_pixels * 100

In [ ]:
fig, ax = plt.subplots(figsize = (8, 5))
ax.hist(total_coverage_pct, bins = 40, color = "#5470b0", edgecolor = "white")
ax.set_title("Cobertura total de las máscaras por imagen (todas las clases)", fontweight = "bold")
ax.set_xlabel("% de los píxeles de la imagen cubiertos")
ax.set_ylabel("Número de imágenes")
plt.tight_layout()
guardar(fig, "cobertura_total")
plt.show()

La distribución obtenida muestra que la mayoría de las imágenes presentan niveles de cobertura intermedios, mientras que las escenas completamente despejadas o totalmente cubiertas por nubes son menos comunes. Esta variabilidad representa un desafío importante para los modelos de segmentación, ya que deben aprender a identificar correctamente tanto regiones pequeñas como extensas dentro de la misma distribución de datos.

#### 3.6.3. Área de la máscara por clase
Ahora, se analiza la cobertura de las máscaras para cada categoría de nube de manera independiente.

In [ ]:
colors = ["#03396c", "#005b96", "#6497b1", "#b3cde0"]

fig, axes = plt.subplots(2, 2, figsize = (12, 8))

for ax, label, color in zip(axes.flat, LABELS, colors):
    label_pct = coverage_pixels[label][coverage_pixels[label] > 0] / total_pixels * 100
    ax.hist(label_pct, bins = 40, color = color, edgecolor = "white")
    ax.set_title(f"{label}", fontweight = "bold")
    ax.set_xlabel("% de los píxeles de la imagen cubiertos")
    ax.set_ylabel("Número de imágenes")

fig.suptitle("Cobertura de la máscara por clase (solo imágenes con esa clase)", fontweight = "bold")
plt.tight_layout()
guardar(fig, "cobertura_por_clase")
plt.show()

De la figura anterior se observa que la cobertura por clase tiene una alta variabilidad en las cuatro categorías. De acuerdo con la tabla de la sección 3.11, cuando la clase está presente Flower es la que ocupa en promedio la mayor área (32 % de la imagen), seguida de Fish (29 %) y Gravel (29 %), mientras que Sugar ocupa la menor (24 %). Las diferencias entre clases son moderadas frente a la dispersión dentro de cada una (desviación de 16 a 20 puntos).

Estas diferencias son relevantes para el entrenamiento, ya que las clases con coberturas más reducidas aportan menos píxeles positivos durante el aprendizaje.

#### 3.6.4. Posición de las máscaras
Para identificar si alguna clase se concentra en una zona de la imagen, se promedian las máscaras de 500 imágenes por clase (cada píxel indica la proporción de imágenes donde está cubierto).


In [ ]:
fig, axes = plt.subplots(1, 4, figsize = (16, 3))

for ax, label in zip(axes, LABELS):
    names = presence_df.index[presence_df[label] == 1].to_series().sample(500, random_state = 44)
    masks = [rle_to_mask(pivot_training_df.loc[name, label], IMG_HEIGHT, IMG_WIDTH)[::20, ::20].copy() for name in names]
    sns.heatmap(np.mean(masks, axis = 0), ax = ax, cmap = "Blues", vmin = 0, vmax = 1,
                cbar = False, xticklabels = False, yticklabels = False)
    ax.set_title(label, fontweight = "bold")

fig.suptitle("Frecuencia de cobertura por posición", fontweight = "bold", fontsize = 16)
plt.tight_layout()
guardar(fig, "cobertura_por_posicion")
plt.show()

Las clases muestran un sesgo leve de posición. Fish se concentra en la zona central-izquierda y Flower en la derecha y la parte inferior, mientras que Gravel y Sugar se distribuyen de forma más uniforme. Este sesgo sugiere aplicar flips para que el modelo no aprenda la posición como atajo.

### 3.7. Necesidades de preprocesamiento
Las imágenes son grandes, por lo que se evalúa cuánto se degradan las máscaras al reducir la resolución (IoU entre la máscara original y la reducida y restaurada, en 30 imágenes), y se calculan la media y desviación estándar por canal para normalizar.


In [ ]:
def resize_iou(mask, factor):
    h, w = mask.shape
    small = mask.reshape(h // factor, factor, w // factor, factor).mean(axis = (1, 3)) > 0.5
    restored = np.repeat(np.repeat(small, factor, axis = 0), factor, axis = 1)
    mask = mask > 0.5
    return (mask & restored).sum() / (mask | restored).sum()


sample_names = presence_df.index.to_series().sample(30, random_state = 44)
resize_ious = {factor: [] for factor in [2, 4, 7]}
for name in sample_names:
    for label in [label for label in LABELS if presence_df.loc[name, label]]:
        mask = rle_to_mask(pivot_training_df.loc[name, label], IMG_HEIGHT, IMG_WIDTH)
        for factor in resize_ious:
            resize_ious[factor].append(resize_iou(mask, factor))

print(pd.Series({f"{IMG_HEIGHT // f}x{IMG_WIDTH // f}": np.mean(v) for f, v in resize_ious.items()},
                name = "IoU promedio").round(3).to_string(), end = "\n\n")

mean_cols = [f"{c}_mean" for c in "RGB"]
std_cols = [f"{c}_std" for c in "RGB"]
norm_mean = image_df[mean_cols].mean().values
norm_std = np.sqrt((image_df[std_cols] ** 2).mean().values + image_df[mean_cols].var(ddof = 0).values)
print(f"Media por canal (RGB): {norm_mean.round(3)}")
print(f"Desviación estándar por canal (RGB): {norm_std.round(3)}")

Reducir las imágenes a 350x525 (un cuarto por lado) conserva prácticamente la máscara original (IoU de 0.993), e incluso 200x300 mantiene 0.988. Por lo tanto, se puede entrenar a resolución reducida con una pérdida mínima de calidad y un costo computacional mucho menor.

Para normalizar se proponen la media (0.261, 0.279, 0.327) y la desviación estándar (0.242, 0.237, 0.222) por canal RGB, calculadas a un cuarto de la resolución. No hay datos faltantes que completar: las clases ausentes en una imagen se representan como una máscara vacía, que es un ejemplo negativo válido.

**Decisión adoptada.** Las imágenes se reducen a 350x525 y se rellenan con un borde negro hasta 352x544 (2 píxeles abajo y 19 a la derecha, con máscara vacía), porque los encoders convolucionales usados reducen la imagen a la mitad cinco veces y requieren dimensiones divisibles por 32. El relleno no deforma ni recorta la imagen. Esta preparación está implementada en la función `preparar_datos()` de `src/datos.py`, que todos los notebooks de modelos usan.

### 3.8. Data augmentation
- **Diversidad:** las escenas cubren distintas condiciones de brillo y tienen franjas sin datos; además, las clases tienen un leve sesgo de posición (3.6.4).
- **Estrategia adoptada (`aumento()` en `src/datos.py`):** flips horizontal y vertical con probabilidad 0.5, una transformación afín suave con probabilidad 0.5 (escala entre 0.9 y 1.1, desplazamiento de hasta 5 % y rotación de hasta 10°) y variaciones leves de brillo y contraste (±10 %). Las transformaciones geométricas se aplican igual a la imagen y a sus máscaras.
- **Qué se evita:** cambios de color fuertes o deformaciones que alteren la textura de las nubes, pues esta define la clase (sección 3.13). Tampoco se usan rotaciones de 90°, porque con entradas de 352x544 cambiarían la forma de la imagen y voltearían la orientación de estructuras alargadas como Fish.
- **Transfer learning:** con unas 5500 imágenes se recomienda usar un encoder preentrenado (por ejemplo, en ImageNet) en lugar de entrenar desde cero. No parece necesario generar datos sintéticos.

### 3.9. División del dataset
Solo el conjunto de entrenamiento tiene máscaras, por lo que la partición entrenamiento/validación se hace sobre este. Se propone una división 80/20 estratificada por la combinación de etiquetas de cada imagen, para mantener el balance de clases, y se verifica que no haya contenido duplicado entre ambas partes.


In [ ]:
pattern = presence_df.astype(str).agg("".join, axis = 1)
val_images = pattern.groupby(pattern).sample(frac = 0.2, random_state = 44).index
split = pd.Series("train", index = presence_df.index, name = "split")
split[val_images] = "val"

leakage = image_df.assign(split = split).groupby("md5")["split"].nunique().gt(1).sum()
print(f"Contenidos duplicados entre train y val: {leakage}\n")

split_summary = presence_df.groupby(split).mean().round(3)
split_summary["n_images"] = split.value_counts()
split_summary

La partición propuesta deja 4438 imágenes para entrenamiento y 1108 para validación, con proporciones de clase casi idénticas en ambas (por ejemplo, Sugar 0.676 en las dos) y sin contenido duplicado entre ellas. Las imágenes de prueba de Kaggle no tienen máscaras públicas, por lo que la evaluación local se hace con el conjunto de validación.

**Partición final adoptada (80/10/10).** La validación se usa para elegir la mejor época, el umbral de decisión y el posprocesado, por lo que la evaluación final necesita imágenes que no participen en ninguna decisión. Se mantiene el 80 % para entrenamiento y el 20 % restante se divide en validación y prueba, con la misma estratificación por combinación de etiquetas y la misma semilla (44):

| Conjunto | Imágenes |
|---|---|
| Entrenamiento | 4438 |
| Validación | 553 |
| Prueba | 555 |

La partición se generó una sola vez con la función `crear_split()` de `src/datos.py` y quedó guardada en `splits/split.csv`, que es el archivo que usan todos los modelos del proyecto. La siguiente tabla verifica que las tres partes conservan las proporciones de cada clase.

In [ ]:
split_final = pd.read_csv(SPLIT_CSV, index_col = 0)["split"].reindex(presence_df.index)
resumen_split = presence_df.groupby(split_final).mean().round(3)
resumen_split["n_imagenes"] = split_final.value_counts()
resumen_split = resumen_split.loc[["train", "val", "test"]]
resumen_split.to_csv(EDA_DIR / "proporciones_por_particion.csv")
resumen_split

Las proporciones de cada clase son prácticamente iguales en entrenamiento, validación y prueba, por lo que las métricas de validación y prueba son representativas del conjunto completo.

### 3.10. Revisión de anotaciones
Además de la inspección visual de las secciones anteriores, se verifica a nivel de píxel si las máscaras de distintas clases pueden solaparse, lo que determina cómo debe modelarse la salida de la segmentación. A continuación se superponen las máscaras sobre una imagen con varias clases.


In [ ]:
def overlapping_pixel_count(row):
    count = np.zeros(total_pixels, dtype = np.uint8)
    for label in LABELS:
        nums = np.array(row[label].split(), dtype = int)
        for s, l in zip(nums[0::2] - 1, nums[1::2]):
            count[s:s + l] += 1
    return (count > 1).sum()


overlap_pct = pivot_training_df[LABELS].apply(overlapping_pixel_count, axis = 1) / total_pixels * 100
has_overlap = overlap_pct > 0

print(f"Imágenes con píxeles en más de una clase: {has_overlap.sum()} de {(presence_df.sum(axis = 1) > 1).sum()} con 2 o más etiquetas")
print(f"Solapamiento en esas imágenes (% de píxeles) - mediana: {overlap_pct[has_overlap].median():.1f}, máximo: {overlap_pct.max():.1f}")

El 79 % de las imágenes con dos o más etiquetas (3322 de 4198) tiene píxeles asignados a más de una clase, con un solapamiento mediano del 11 % de la imagen y de hasta 82 %. Por lo tanto, las clases no son mutuamente excluyentes a nivel de píxel.

In [ ]:
img_name = "0432314.jpg"
images_path = IMAGES_PATH
plot_image(img_name, images_path)

La inspección permitió verificar la consistencia de las anotaciones: las máscaras ubican correctamente los patrones nubosos presentes en las imágenes, aunque de forma aproximada, pues son uniones de regiones rectangulares que no siguen el contorno de las nubes (ver sección 3.12).

### 3.11. Métricas exploratorias
Se resumen, por clase, el brillo, la nitidez y la cobertura de la máscara (media y desviación estándar entre las imágenes donde aparece la clase).


In [ ]:
rows = {}
for label in LABELS:
    present = presence_df[label] == 1
    subset = image_df.loc[present]
    coverage = coverage_pixels.loc[present, label] / total_pixels * 100
    rows[label] = {
        "brillo_media": subset["brightness"].mean(), "brillo_std": subset["brightness"].std(),
        "nitidez_media": subset["sharpness"].mean(), "nitidez_std": subset["sharpness"].std(),
        "cobertura_media_%": coverage.mean(), "cobertura_std_%": coverage.std(),
    }
pd.DataFrame(rows).T.round(4)

Las clases son muy similares en brillo (0.283 a 0.289) y nitidez (0.044 a 0.050), con desviaciones del mismo orden dentro de cada clase, por lo que estas características globales no las separan y el modelo debe apoyarse en la textura y la forma de las nubes. La cobertura media de la máscara cuando la clase está presente va de 24 % (Sugar) a 32 % (Flower), con alta variabilidad (desviación de 16 a 20 puntos).

### 3.12. Geometría de las etiquetas por clase
Se describe la forma de las regiones anotadas en una muestra de 400 imágenes por clase: cuántas regiones separadas tiene cada máscara, su tamaño, su relación de aspecto y qué tan rectangulares son (extensión = área de la región / área de su caja envolvente; 1 = rectángulo perfecto).

In [ ]:
F = 4                      # se trabaja a 1/4 de resolución (350x525)
N_POR_CLASE = 400          # imágenes muestreadas por clase
SEED = 44


def regiones(mask_bool):
    """Devuelve una fila por región conectada de la máscara."""
    n, _, stats, _ = cv2.connectedComponentsWithStats(mask_bool.astype(np.uint8), connectivity = 8)
    total = mask_bool.size
    filas = []
    for x, y, w, h, area in stats[1:]:          # stats[0] es el fondo
        if area < 0.002 * total:                # descarta residuos menores al 0.2 % de la imagen
            continue
        filas.append({"area_pct": 100 * area / total, "aspecto": w / h, "extension": area / (w * h)})
    return filas


geo_regiones, geo_mascaras = [], []
for label in LABELS:
    nombres = presence_df.index[presence_df[label] == 1].to_series().sample(N_POR_CLASE, random_state = SEED)
    for nombre in nombres:
        mask = rle_to_mask(pivot_training_df.loc[nombre, label], IMG_HEIGHT, IMG_WIDTH)[::F, ::F] > 0.5
        filas = regiones(mask)
        geo_mascaras.append({"clase": label, "n_regiones": len(filas)})
        geo_regiones += [{"clase": label, **f} for f in filas]

geo_regiones = pd.DataFrame(geo_regiones)
geo_mascaras = pd.DataFrame(geo_mascaras)

resumen_geo = geo_regiones.groupby("clase")[["area_pct", "aspecto", "extension"]].median()
resumen_geo["regiones_por_imagen"] = geo_mascaras.groupby("clase")["n_regiones"].mean()
resumen_geo["pct_casi_rectangular"] = geo_regiones.assign(r = geo_regiones["extension"] > 0.95).groupby("clase")["r"].mean() * 100
resumen_geo.round(2).to_csv(f"{EDA_DIR}/geometria_por_clase.csv")

fig, axes = plt.subplots(1, 3, figsize = (16, 4))
for ax, col, titulo in zip(axes, ["area_pct", "aspecto", "extension"],
                           ["Área de la región (% de la imagen)", "Relación de aspecto (ancho / alto)",
                            "Extensión (1 = rectángulo perfecto)"]):
    sns.boxplot(data = geo_regiones, x = "clase", y = col, ax = ax, color = "#6497b1", showfliers = False)
    ax.set_title(titulo, fontweight = "bold")
    ax.set_xlabel("")
plt.tight_layout()
plt.savefig(f"{EDA_DIR}/geometria_por_clase.png", dpi = 120)
plt.show()

resumen_geo.round(2)

**Resultado** (medianas por clase; ejecutado en Kaggle sobre una muestra de 400 imágenes por clase, tabla en `artifacts/eda/geometria_por_clase.csv`):

| Clase | Área de la región (%) | Aspecto (ancho/alto) | Extensión | Regiones por imagen | % casi rectangular |
|---|---|---|---|---|---|
| Fish | 16.66 | 1.56 | 0.96 | 1.40 | 54.63 |
| Flower | 20.04 | 1.06 | 0.93 | 1.37 | 44.99 |
| Gravel | 15.89 | 1.21 | 0.98 | 1.36 | 58.82 |
| Sugar | 12.39 | 1.29 | 0.98 | 1.47 | 59.01 |

![Geometría de las etiquetas por clase](../artifacts/eda/geometria_por_clase.png)

Las regiones anotadas son casi rectangulares en las cuatro clases (extensión mediana entre 0.93 y 0.98) y cada máscara tiene en promedio menos de dos regiones. Existen diferencias leves: Fish presenta regiones más apaisadas (relación de aspecto 1.56), Flower regiones más cuadradas y de mayor tamaño (1.06 y 20 % de la imagen) y Sugar las más pequeñas (12 %). Sin embargo, las distribuciones se solapan ampliamente, por lo que la geometría de la etiqueta no permite distinguir una clase de otra. La diferencia entre clases debe buscarse en el contenido de la región, es decir, en la textura y la organización de las nubes.

### 3.13. Textura dentro de la máscara por clase
Para cada clase se toma la zona que pertenece solo a esa clase (sin solapamiento con otras), se separa nube de océano con el umbral de Otsu y se describen las nubes que quedan dentro: fracción nubosa, número y tamaño de los objetos nubosos, concentración de la nube en el objeto mayor, elongación y densidad de bordes.

In [ ]:
def cargar_gris(nombre):
    with Image.open(os.path.join(IMAGES_PATH, nombre)) as img:
        img = img.convert("L").resize((IMG_WIDTH // F, IMG_HEIGHT // F), Image.BILINEAR)
        return np.asarray(img, dtype = np.uint8)


def textura(nombre, label):
    gris = cargar_gris(nombre)
    valido = gris > 5                                   # excluye las franjas negras sin datos
    if valido.sum() < 0.2 * gris.size:
        return None
    masks = {l: rle_to_mask(pivot_training_df.loc[nombre, l], IMG_HEIGHT, IMG_WIDTH)[::F, ::F] > 0.5 for l in LABELS}
    otras = np.zeros_like(valido)
    for l in LABELS:
        if l != label:
            otras |= masks[l]
    zona = masks[label] & ~otras & valido               # zona exclusiva de la clase
    if zona.sum() < 0.02 * gris.size:
        return None

    umbral, _ = cv2.threshold(gris[valido].reshape(-1, 1), 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    nube = (gris > umbral) & zona
    bordes = cv2.Canny(gris, 50, 150) > 0

    n, etiquetas, stats, _ = cv2.connectedComponentsWithStats(nube.astype(np.uint8), connectivity = 8)
    areas = stats[1:, cv2.CC_STAT_AREA]
    areas_ok = areas[areas >= 4]                        # ignora objetos de menos de 4 píxeles
    elong = np.nan
    if len(areas) and areas.max() >= 20:
        mayor = (etiquetas == 1 + int(areas.argmax())).astype(np.uint8)
        contornos, _ = cv2.findContours(mayor, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
        (_, _), (w, h), _ = cv2.minAreaRect(max(contornos, key = cv2.contourArea))
        if min(w, h) > 0:
            elong = max(w, h) / min(w, h)

    return {
        "clase": label,
        "fraccion_nubosa": nube.sum() / zona.sum(),
        "objetos_por_10k_px": 1e4 * len(areas_ok) / zona.sum(),
        "area_mediana_objeto_px": float(np.median(areas_ok)) if len(areas_ok) else np.nan,
        "pct_nube_en_objeto_mayor": 100 * areas.max() / nube.sum() if nube.sum() else np.nan,
        "elongacion_objeto_mayor": elong,
        "densidad_bordes": bordes[zona].mean(),
        "brillo_medio": gris[zona].mean() / 255,
        "brillo_std": gris[zona].std() / 255,
    }


tareas = []
for label in LABELS:
    nombres = presence_df.index[presence_df[label] == 1].to_series().sample(N_POR_CLASE, random_state = SEED)
    tareas += [(nombre, label) for nombre in nombres]

with ThreadPoolExecutor() as executor:
    filas = list(executor.map(lambda t: textura(*t), tareas))
tex_df = pd.DataFrame([f for f in filas if f is not None])
print(tex_df["clase"].value_counts().rename("zonas exclusivas analizadas").to_string())

resumen_tex = tex_df.groupby("clase").median(numeric_only = True).round(3)
resumen_tex.to_csv(f"{EDA_DIR}/textura_por_clase.csv")

cols = ["fraccion_nubosa", "objetos_por_10k_px", "area_mediana_objeto_px",
        "pct_nube_en_objeto_mayor", "elongacion_objeto_mayor", "densidad_bordes"]
fig, axes = plt.subplots(2, 3, figsize = (16, 8))
for ax, col in zip(axes.flat, cols):
    sns.boxplot(data = tex_df, x = "clase", y = col, ax = ax, color = "#6497b1", showfliers = False)
    ax.set_title(col.replace("_", " "), fontweight = "bold")
    ax.set_xlabel("")
plt.tight_layout()
plt.savefig(f"{EDA_DIR}/textura_por_clase.png", dpi = 120)
plt.show()

resumen_tex

**Resultado** (medianas por clase; se analizaron 357 zonas exclusivas de Fish, 380 de Flower, 342 de Gravel y 338 de Sugar; tabla completa, con elongación y densidad de bordes, en `artifacts/eda/textura_por_clase.csv`):

| Clase | Fracción nubosa | Objetos por 10 000 px | Área mediana del objeto (px) | % de la nube en el objeto mayor |
|---|---|---|---|---|
| Fish | 0.323 | 18.638 | 10.0 | 52.713 |
| Flower | 0.329 | 18.828 | 13.0 | 34.177 |
| Gravel | 0.198 | 33.261 | 9.5 | 26.914 |
| Sugar | 0.115 | 24.434 | 9.0 | 28.067 |

![Textura dentro de la máscara por clase](../artifacts/eda/textura_por_clase.png)

A diferencia de la geometría de la etiqueta, el contenido de las regiones sí distingue las clases. Sugar tiene la menor fracción nubosa (0.12), es decir, nubes escasas y dispersas. Gravel es la más fragmentada (33 objetos por cada 10 000 píxeles) con una cobertura intermedia (0.20). Flower y Fish tienen la mayor cobertura (0.33 y 0.32) y menos objetos, pero se diferencian en su organización: en Fish más de la mitad de la nube pertenece a una sola estructura conectada (53 %) y alargada, mientras que en Flower la nube se reparte en masas más compactas (34 %). Por lo tanto, lo que diferencia una clase de otra no es la forma de la región anotada sino la textura y la organización espacial de las nubes en su interior.

**Limitaciones de este análisis.** Las medianas difieren, pero las distribuciones se solapan: ninguna variable por sí sola clasifica una región, y por eso se requiere un modelo que combine textura y contexto. El análisis se hizo a un cuarto de la resolución y con un umbral global de Otsu, de modo que las nubes más pequeñas de Sugar pueden quedar por debajo de un píxel.

### 3.14. Conclusiones de la exploración
**Observaciones principales**
- El conjunto es consistente: 5546 imágenes de 2100x1400, sin duplicados, sin archivos corruptos y con todas las máscaras RLE válidas.
- El problema es multi-etiqueta también a nivel de píxel: el 79 % de las imágenes con dos o más etiquetas tiene píxeles asignados a más de una clase. Por eso cada clase se predice con una máscara binaria independiente (sigmoide por canal) y no con una única clase por píxel.
- Las etiquetas son uniones de rectángulos (sección 3.12). La anotación de origen es de tipo caja, pero la tarea, la salida del modelo y la métrica son de segmentación por píxel.
- Lo que diferencia las clases es la textura y la organización de las nubes dentro de la región (sección 3.13), no la geometría de la etiqueta ni el brillo o la nitidez globales (sección 3.11).

**Riesgos para el modelado**
- Etiquetas gruesas y ruidosas: los bordes de las máscaras no son verdad de terreno, lo que limita el Dice alcanzable.
- Desbalance moderado entre clases (Sugar 68 % de las imágenes, Flower 43 %).
- Franjas negras sin datos en algunas imágenes y sesgo leve de posición de las clases.

**Plan de preprocesamiento y aumento de datos**
- Reducción a 350x525 con relleno a 352x544 y normalización con la media y desviación por canal del conjunto.
- Partición 80/10/10 estratificada y compartida por todos los modelos (`splits/split.csv`).
- Aumento con flips, transformaciones geométricas suaves y variaciones leves de brillo y contraste, sin alterar la textura y sin rotaciones de 90°.
- Encoder preentrenado y pérdida focal más Dice para atender el tamaño del conjunto y el desbalance.